In [ ]:
!git clone https://github.com/ajurcik/MLPrague-2026-test.git
!pip install ./MLPrague-2026-test
!pip install -r MLPrague-2026-test/requirements.txt

# Unsupervised Anomaly Detection on Graphs — Hands-on Workshop

In this notebook you will apply several anomaly detection methods to the **YelpChi** fraud-review dataset.  
The dataset contains Yelp reviews with hand-crafted features and a binary `spam` label. Reviews are connected through edges, forming a graph.

**Approaches we will train & compare:**
* **Isolation Forest** — a classical tree-based anomaly detector
* **Graph Auto Encoder** — mix of GNN and autoencoder
* **Heterogeneous Graph Auto Encoder** - GAE version with heterogeneous edges

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display, HTML
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import minmax_scale

from ml_prague_2026.evaluation import compare_models, evaluate_model
from ml_prague_2026.utils import prepare_yelp_chi_tabular_data, sample_pinsage_neighbors, sample_pinsage_neighbors_hetero

import torch
import torch.nn as tnn
import torch_geometric.nn as nn
import torch_geometric.utils as utils
import torch.nn.functional as F
from torch_geometric.utils import to_undirected
from torch_geometric.data import Data, HeteroData
from torch_geometric.nn import GCNConv, HeteroConv
from torch_geometric.seed import seed_everything

from sklearn.preprocessing import StandardScaler


seed_everything(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
YELP_CHI_PATH = 'MLPrague-2026-test/data/yelpchi.parquet'

YELP_CHI_EDGE_RSR_PATH = 'MLPrague-2026-test/data/yelpchi_rsr.npy'
YELP_CHI_EDGE_RTR_PATH = 'MLPrague-2026-test/data/yelpchi_rtr.npy'
YELP_CHI_EDGE_RUR_PATH = 'MLPrague-2026-test/data/yelpchi_rur.npy'

EXAMPLES_NORMAL = [7696, 37814, 12186]
EXAMPLES_SPAM = [44708, 43100, 42660]

---
## 1. Data Loading

We start by loading the preprocessed YelpChi dataset (reviews with features) and the three edge lists that connect reviews sharing the same restaurant, timestamp, or user.

In [ ]:
yelp_chi = pd.read_parquet(YELP_CHI_PATH)
yelp_chi.head(2)

In [ ]:
edges = np.concatenate((
    np.load(YELP_CHI_EDGE_RSR_PATH, allow_pickle=True),
    np.load(YELP_CHI_EDGE_RTR_PATH, allow_pickle=True),
    np.load(YELP_CHI_EDGE_RUR_PATH, allow_pickle=True),
), axis=0)

In [ ]:
# Load train/test split from file (to ensure consistency with supervised notebook)
data_split = np.load('MLPrague-2026-test/data/yelpchi_split.npz')
train_idx = np.concatenate([data_split['train'], data_split['val']])
test_idx = data_split['test']

---
## 2. Isolation Forest

[Isolation Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html) is a tree-based anomaly detector that works on tabular features **without** graph structure.  
It isolates anomalies by randomly partitioning features — anomalies require fewer splits, resulting in shorter average path lengths.

<img src="https://raw.githubusercontent.com/ajurcik/MLPrague-2026-test/refs/heads/master/images/iso_forest.png" width="600">

In [ ]:
# Prepare train/test features and labels for Yelp-Chi in tabular form
# add_degree_feature adds a degree features to the data file for each edge type
X_train, X_test, y_train, y_test = prepare_yelp_chi_tabular_data(yelp_chi, train_idx, test_idx, add_degree_feature=False)

We use the known **contamination rate** (fraction of spam) so the model threshold matches the true anomaly ratio.

In [ ]:
# Contamination approximates the fraction of anomalies (spam) in the dataset
contamination = y_train.mean()
print(f'Contamination (proportion of anomalies): {contamination:.4f}')

#### TASK 2.A: Experiment with contamination

In [ ]:
# TASK 2.A: experiment with the contamination parameter
# - Try different values for contamination (e.g., 0.1, 0.2...)
# - Observe how it affects the recall and precision of the model on anomaly class

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1,
)

iso_forest.fit(X_train)

In [ ]:
# IsolationForest returns -1 for anomalies, 1 for normal
y_pred_test = (iso_forest.predict(X_test) == -1).astype(int)

# Compute anomaly scores — negate score_samples so that higher = more anomalous
y_scores_test = -iso_forest.score_samples(X_test)
y_scores_test

In [ ]:
# Sklearn classification report
print(classification_report(y_test, y_pred_test))

# Evaluate
isolation_forest_results = evaluate_model('isolation_forest', y_test, y_pred_test, y_scores_test)

### Isolation forest with degree feature
Let's try adding the number of connected neighbors as a graph feature.

In [ ]:
edge_paths = {
    "rur": YELP_CHI_EDGE_RUR_PATH,
    "rsr": YELP_CHI_EDGE_RSR_PATH,
    "rtr": YELP_CHI_EDGE_RTR_PATH,
}

X_train_graph, X_test_graph, y_train_graph, y_test_graph = prepare_yelp_chi_tabular_data(yelp_chi, train_idx, test_idx, add_degree_feature=True, edge_paths=edge_paths)

In [ ]:
iso_forest_graph = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1,
)

iso_forest_graph.fit(X_train_graph)

In [ ]:
# IsolationForest returns -1 for anomalies, 1 for normal
y_pred_test_graph = (iso_forest_graph.predict(X_test_graph) == -1).astype(int)

# Compute anomaly scores — negate score_samples so that higher = more anomalous
y_scores_test_graph = -iso_forest_graph.score_samples(X_test_graph)
y_scores_test_graph

In [ ]:
# Evaluate
isolation_forest_graph_results = evaluate_model('isolation_forest_graph', y_test_graph, y_pred_test_graph, y_scores_test_graph)

### Compare isolation forest models

In [ ]:
models_results = compare_models([
    isolation_forest_results,
    isolation_forest_graph_results,
])

---
## 3. Graph auto encoder

GAE is autoencoder that uses a graph neural network as an encoder that learns node representations that reconstruct the original node features. Nodes with high reconstruction error (i.e., those the model fails to reconstruct well) are flagged as anomalies.

<img src="https://raw.githubusercontent.com/ajurcik/MLPrague-2026-test/refs/heads/master/images/gae.png" width="600">

We use the custom Pytorch Geometric implementation.

#### Building the PyTorch Geometric Graph

To use GNN-based detectors we need a `torch_geometric.Data` object that bundles:
- **`x`** — node feature matrix of shape `(num_nodes, num_features)`
- **`edge_index`** — edge list as a `(2, num_edges)` tensor with contiguous node indices
- **`y`** — ground-truth labels (used only for evaluation, not for training)

In [ ]:
# Create edge index tensor
edge_index = torch.tensor(edges, dtype=torch.long)
if edge_index.shape[0] != 2:
    edge_index = edge_index.t()
edge_index = edge_index.contiguous()

# Add undirected edges (since the graph is undirected)
edge_index = utils.to_undirected(edge_index)

# Extract node features — select all columns that start with 'f_'
feature_cols = [c for c in yelp_chi.columns if c.startswith('f_')]
x = torch.tensor(yelp_chi[feature_cols].values, dtype=torch.float)

# Extract labels from the 'spam' column
y = torch.tensor(yelp_chi['spam'].values, dtype=torch.long)

# Create the PyTorch Geometric Data object with x, edge_index, and y
pytorch_data = Data(x=x, edge_index=edge_index, y=y)
pytorch_data = pytorch_data.to(device)

pytorch_data

In [ ]:
# Apply train/test split to the PyG data object via masks
pytorch_data.train_mask = torch.zeros(pytorch_data.num_nodes, dtype=torch.bool)
pytorch_data.test_mask = torch.zeros(pytorch_data.num_nodes, dtype=torch.bool)
pytorch_data.train_mask[train_idx] = True
pytorch_data.test_mask[test_idx] = True
pytorch_data

### Graph Auto Encoder baseline
Train custom GAE without any data preprocessing.

#### Model

In [ ]:
def mask_features(x, p_mask=0.3, mask_value=-1.0):
    """Returns corrupted x and a boolean mask of which positions were masked."""
    mask = torch.rand_like(x) < p_mask
    x_masked = x.clone()
    x_masked[mask] = mask_value
    return x_masked, mask


class GCNEncoder(tnn.Module):
    """Multi-layer GCN encoder that produces latent node embeddings."""

    def __init__(self, in_channels, hidden_channels, latent_channels, num_layers=2, dropout=0.3):
        super().__init__()
        self.convs = tnn.ModuleList()
        self.dropout = dropout

        if num_layers == 1:
            self.convs.append(GCNConv(in_channels, latent_channels))
        else:
            self.convs.append(GCNConv(in_channels, hidden_channels))
            for _ in range(num_layers - 2):
                self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.convs.append(GCNConv(hidden_channels, latent_channels))


    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.leaky_relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x


class FeatureDecoder(tnn.Module):
    """MLP decoder that reconstructs node features from latent embeddings."""

    def __init__(self, latent_channels, hidden_channels, out_channels):
        super().__init__()
        self.fc1 = tnn.Linear(latent_channels, hidden_channels)
        self.fc2 = tnn.Linear(hidden_channels, out_channels)

    def forward(self, z):
        z = F.leaky_relu(self.fc1(z))
        return self.fc2(z)


class CustomGAE(tnn.Module):
    def __init__(self, in_channels, hidden_channels, latent_channels, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = GCNEncoder(in_channels, hidden_channels, latent_channels, num_layers, dropout)
        self.decoder = FeatureDecoder(latent_channels, hidden_channels, in_channels)

    def forward(self, x, edge_index):
        z = self.encoder(x, edge_index)
        x_hat = self.decoder(z)
        return x_hat

In [ ]:
def train_custom_gae(
    model,
    data,
    num_epochs=200,
    lr=1e-3,
    weight_decay=1e-5,
    p_mask=0.3,
    mask_value=-1.0,
    log_every=10,
    verbose=True,
):
    """ Train a CustomGAE model with masked feature reconstruction. """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_mask = data.train_mask.unsqueeze(1)
    losses = []

    model.train()
    for epoch in range(1, num_epochs + 1):
        optimizer.zero_grad()

        x_masked, mask = mask_features(data.x, p_mask=p_mask, mask_value=mask_value)
        x_hat = model(x_masked, data.edge_index)

        node_mask = train_mask & mask
        loss = F.mse_loss(x_hat[node_mask], data.x[node_mask])

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if verbose and log_every and (epoch % log_every == 0 or epoch == 1):
            print(f"Epoch {epoch:3d}/{num_epochs} | Train Loss: {loss.item():.6f}")

    return {"losses": losses}


In [ ]:
in_channels = pytorch_data.x.size(1)
custom_gae = CustomGAE(
    in_channels=in_channels,
    hidden_channels=64,
    latent_channels=32,
    num_layers=2,
    dropout=0.3,
).to(device)

history = train_custom_gae(custom_gae, pytorch_data.to(device), num_epochs=200)

In [ ]:
# Compute per-node reconstruction error as anomaly score
custom_gae.eval()
with torch.no_grad():
    x_hat = custom_gae(pytorch_data.x, pytorch_data.edge_index)
    recon_error = ((pytorch_data.x - x_hat) ** 2).mean(dim=1)

# Convert anomaly scores to numpy
anomaly_scores = recon_error.cpu().numpy()

# Apply threshold using contamination ratio (same as other models)
threshold = np.percentile(anomaly_scores[pytorch_data.train_mask.cpu().numpy()], 100 * (1 - contamination))
custom_gae_preds = (anomaly_scores >= threshold).astype(int)

# Normalise scores to [0, 1] for probability-like output
custom_gae_preds_proba = minmax_scale(anomaly_scores)

In [ ]:
custom_gae_results = evaluate_model(
    'Custom GAE',
    pytorch_data.y[pytorch_data.test_mask].cpu().numpy(),
    custom_gae_preds[pytorch_data.test_mask.cpu().numpy()],
    custom_gae_preds_proba[pytorch_data.test_mask.cpu().numpy()],
)

#### TASK 3.A: Change the anomaly-score function

We currently score each node by the **mean squared reconstruction error across all features**:
```python
recon_error = ((pytorch_data.x - x_hat) ** 2).mean(dim=1)
```
This treats every feature equally and averages out the signal. Try a few alternatives and see how precision / recall / F1 change. Things you can try:

- **Worst-feature error** — use `.max(dim=1).values` instead of `.mean(dim=1)`. Flags a node as anomalous if *any single feature* is badly reconstructed.
- **L1 (absolute) error** — `(pytorch_data.x - x_hat).abs().mean(dim=1)`. Less sensitive to a few extreme deviations.
- **Top-k error** — average of the `k` largest squared errors per node, e.g. `((pytorch_data.x - x_hat) ** 2).topk(5, dim=1).values.mean(dim=1)`.

Re-run the scoring + evaluation cells and compare with the baseline.

In [ ]:
# TASK 3.A: Change the anomaly-score function
# Only change recon_error variable and re-run this cell and evaluation cell

# Compute per-node reconstruction error as anomaly score
custom_gae.eval()
with torch.no_grad():
    x_hat = custom_gae(pytorch_data.x, pytorch_data.edge_index)
    recon_error = ((pytorch_data.x - x_hat) ** 2).mean(dim=1)           # CHANGE THIS ROW

# Convert anomaly scores to numpy
anomaly_scores = recon_error.cpu().numpy()

# Apply threshold using contamination ratio (same as other models)
threshold = np.percentile(anomaly_scores[pytorch_data.train_mask.cpu().numpy()], 100 * (1 - contamination))
custom_gae_preds = (anomaly_scores >= threshold).astype(int)

# Normalise scores to [0, 1] for probability-like output
custom_gae_preds_proba = minmax_scale(anomaly_scores)

In [ ]:
custom_gae_results = evaluate_model(
    'Custom GAE',
    pytorch_data.y[pytorch_data.test_mask].cpu().numpy(),
    custom_gae_preds[pytorch_data.test_mask.cpu().numpy()],
    custom_gae_preds_proba[pytorch_data.test_mask.cpu().numpy()],
)

### Graph Auto Encoder edge sampling
Train custom GAE with edge sampling technique inspired by paper [PinSAGE](https://arxiv.org/abs/1806.01973).

The core idea of PinSAGE importance-based neighbor sampling:
* **High-degree nodes** — uniform sampling from a node with millions of neighbors gives you noise.
* **All neighbors are not equally informative** — some are structurally much more "central" to the target node than others.

How it works in short:

We define each node's neighborhood by running short random walks from it, taking the top-K most-visited neighbors.

<img src="https://raw.githubusercontent.com/ajurcik/MLPrague-2026-test/refs/heads/master/images/pinsage.png" width="600">

#### PinSAGE neighbor sampling

In [ ]:
pruned_edge_index, edge_weight = sample_pinsage_neighbors(
    pytorch_data.edge_index,
    num_nodes=pytorch_data.num_nodes,
    num_neighbors=15,
    num_walks=300,
    walk_length=3,
    restart_prob=0.5,
    return_weights=True,
    seed=42
)

pytorch_data_sampled = pytorch_data.clone()

pruned_edge_index, edge_weight = to_undirected(
    pruned_edge_index, edge_attr=edge_weight, reduce='mean'
)
pytorch_data_sampled.edge_index = pruned_edge_index
pytorch_data_sampled.edge_weight = edge_weight

#### Model

In [ ]:
# Instantiate and train the Custom GAE with PinSAGE sampling
in_channels = pytorch_data.x.size(1)
custom_gae_pinsage = CustomGAE(
    in_channels=in_channels,
    hidden_channels=64,
    latent_channels=32,
    num_layers=2,
    dropout=0.3,
).to(device)

history = train_custom_gae(custom_gae_pinsage, pytorch_data_sampled.to(device), num_epochs=200)

In [ ]:
# Compute per-node reconstruction error as anomaly score
custom_gae_pinsage.eval()
with torch.no_grad():
    x_hat = custom_gae_pinsage(pytorch_data_sampled.x, pytorch_data_sampled.edge_index)
    recon_error = ((pytorch_data_sampled.x - x_hat) ** 2).mean(dim=1)

# Convert anomaly scores to numpy
anomaly_scores = recon_error.cpu().numpy()

# Apply threshold using contamination ratio (same as other models)
threshold = np.percentile(anomaly_scores[pytorch_data_sampled.train_mask.cpu().numpy()], 100 * (1 - contamination))
custom_gae_pinsage_preds = (anomaly_scores >= threshold).astype(int)

# Normalise scores to [0, 1] for probability-like output
custom_gae_pinsage_preds_proba = minmax_scale(anomaly_scores)

In [ ]:
custom_gae_pinsage_results = evaluate_model(
    'Custom GAE with neighbor sampling',
    pytorch_data_sampled.y[pytorch_data_sampled.test_mask].cpu().numpy(),
    custom_gae_pinsage_preds[pytorch_data_sampled.test_mask.cpu().numpy()],
    custom_gae_pinsage_preds_proba[pytorch_data_sampled.test_mask.cpu().numpy()],
)

---
## 4. Heterogeneous graph auto encoder

So far we have collapsed every relation in YelpChi into a single homogeneous edge type. Treating all three relations identically can discards useful signals.

We implement this in PyTorch Geometric using `HeteroData` to store the per-relation `edge_index` tensors. The encoder produces a single embedding per review node by summing the per-relation messages, and a small MLP decoder reconstructs the original node features.

We again apply **PinSAGE-style neighbor sampling**, but now per edge type.

In [ ]:
edge_type_map = {
    'rsr': np.load(YELP_CHI_EDGE_RSR_PATH, allow_pickle=True),
    'rtr': np.load(YELP_CHI_EDGE_RTR_PATH, allow_pickle=True),
    'rur': np.load(YELP_CHI_EDGE_RUR_PATH, allow_pickle=True),
}

# Build a HeteroData object with separate edge types
hetero_data = HeteroData()

# Node features and labels (single node type: "review")
feature_cols = [c for c in yelp_chi.columns if c.startswith('f_')]
hetero_data['review'].x = torch.tensor(yelp_chi[feature_cols].values, dtype=torch.float)
hetero_data['review'].y = torch.tensor(yelp_chi['spam'].values, dtype=torch.long)

# Train / test masks
hetero_data['review'].train_mask = torch.zeros(hetero_data['review'].x.size(0), dtype=torch.bool)
hetero_data['review'].test_mask = torch.zeros(hetero_data['review'].x.size(0), dtype=torch.bool)
hetero_data['review'].train_mask[train_idx] = True
hetero_data['review'].test_mask[test_idx] = True

# Load each relation as a separate edge type, made undirected.
for rel_name, edges_arr in edge_type_map.items():
    ei = torch.tensor(edges_arr, dtype=torch.long)
    if ei.shape[0] != 2:
        ei = ei.t()
    ei = utils.to_undirected(ei.contiguous())
    hetero_data['review', rel_name, 'review'].edge_index = ei

hetero_data = hetero_data.to(device)
print(hetero_data)

#### PinSAGE heterogeneous neighbor sampling

In [ ]:
# hetero_data = sample_pinsage_neighbors_hetero(
#     hetero_data,
#     node_type='review',
#     num_neighbors=10,
#     num_walks=300,
#     walk_length=3,
#     restart_prob=0.5,
#     verbose=True,
#     seed=42
# )

In [ ]:
# Load pre-calculated data due to time constraints
hetero_data = torch.load('MLPrague-2026-test/data/hetero_data.pt', weights_only=False)
hetero_data

#### Model

In [ ]:
class HeteroEncoder(tnn.Module):
    """Multi-layer heterogeneous encoder using HeteroConv with SAGEConv per edge type."""

    def __init__(self, in_channels, hidden_channels, latent_channels, edge_types, num_layers=2, dropout=0.3):
        super().__init__()
        self.convs = tnn.ModuleList()
        self.dropout = dropout

        # First layer
        first_conv_dict = {et: GCNConv(in_channels, latent_channels if num_layers == 1 else hidden_channels) for et in edge_types}
        self.convs.append(HeteroConv(first_conv_dict, aggr='mean'))

        # Middle layers
        for _ in range(num_layers - 2):
            mid_conv_dict = {et: GCNConv(hidden_channels, hidden_channels) for et in edge_types}
            self.convs.append(HeteroConv(mid_conv_dict, aggr='mean'))

        # Last layer
        if num_layers > 1:
            last_conv_dict = {et: GCNConv(hidden_channels, latent_channels) for et in edge_types}
            self.convs.append(HeteroConv(last_conv_dict, aggr='mean'))

    def forward(self, x_dict, edge_index_dict):
        for conv in self.convs[:-1]:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: F.leaky_relu(x) for key, x in x_dict.items()}
            x_dict = {key: F.dropout(x, p=self.dropout, training=self.training) for key, x in x_dict.items()}
        x_dict = self.convs[-1](x_dict, edge_index_dict)
        return x_dict


class HeteroGAE(tnn.Module):
    """Heterogeneous Graph Autoencoder: HeteroConv encoder + feature-reconstruction decoder."""

    def __init__(self, in_channels, hidden_channels, latent_channels, edge_types, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = HeteroEncoder(in_channels, hidden_channels, latent_channels, edge_types, num_layers, dropout)
        # Reuse the same MLP decoder for the single node type
        self.decoder = FeatureDecoder(latent_channels, hidden_channels, in_channels)

    def forward(self, x_dict, edge_index_dict):
        z_dict = self.encoder(x_dict, edge_index_dict)
        x_hat = self.decoder(z_dict['review'])
        return x_hat

In [ ]:
def train_hetero_gae(
    model,
    data,
    target_node_type='review',
    num_epochs=150,
    lr=1e-3,
    weight_decay=1e-5,
    p_mask=0.3,
    mask_value=-1.0,
    log_every=10,
    verbose=True,
):
    """ Train a HeteroGAE model with masked feature reconstruction on a target node type. """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    edge_types = list(data.edge_types)
    edge_index_dict = {et: data[et].edge_index for et in edge_types}
    target_x = data[target_node_type].x
    train_mask = data[target_node_type].train_mask.unsqueeze(1)
    device = target_x.device

    losses = []

    model.train()
    for epoch in range(1, num_epochs + 1):
        optimizer.zero_grad()

        x_masked, mask = mask_features(target_x, p_mask=p_mask, mask_value=mask_value)
        x_dict = {target_node_type: x_masked}

        x_hat = model(x_dict, edge_index_dict)

        node_mask = train_mask & mask
        if node_mask.any():
            loss = F.mse_loss(x_hat[node_mask], target_x[node_mask])
        else:
            loss = torch.tensor(0.0, device=device)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if verbose and log_every and (epoch % log_every == 0 or epoch == 1):
            print(f"Epoch {epoch:3d}/{num_epochs} | Train Loss: {loss.item():.6f}")

    return {'losses': losses}

In [ ]:
edge_types = list(hetero_data.edge_types)
in_channels = hetero_data['review'].x.size(1)

hetero_gae = HeteroGAE(
    in_channels=in_channels,
    hidden_channels=64,
    latent_channels=16,
    edge_types=edge_types,
    num_layers=3,
    dropout=0.3,
).to(device)

history_hetero_gae = train_hetero_gae(hetero_gae, hetero_data.to(device), num_epochs=150)

In [ ]:
# Compute per-node reconstruction error as anomaly score
hetero_gae.eval()
with torch.no_grad():
    x_dict = {'review': hetero_data['review'].x}
    edge_index_dict = {et: hetero_data[et].edge_index for et in edge_types}
    x_hat = hetero_gae(x_dict, edge_index_dict)
    recon_error = ((hetero_data['review'].x - x_hat) ** 2).mean(dim=1)

anomaly_scores_hetero = recon_error.cpu().numpy()

threshold_hetero = np.percentile(anomaly_scores_hetero[hetero_data['review'].train_mask.cpu().numpy()], 100 * (1 - contamination))
hetero_gae_preds = (anomaly_scores_hetero >= threshold_hetero).astype(int)

hetero_gae_preds_proba = minmax_scale(anomaly_scores_hetero)

In [ ]:
hetero_gae_results = evaluate_model(
    'Hetero GAE with neighbor sampling',
    hetero_data['review'].y[hetero_data['review'].test_mask].cpu().numpy(),
    hetero_gae_preds[hetero_data['review'].test_mask.cpu().numpy()],
    hetero_gae_preds_proba[hetero_data['review'].test_mask.cpu().numpy()],
)

---
## 5. Final comparison

Let's compare all models side-by-side.

In [ ]:
models_results = compare_models([
    isolation_forest_results,
    isolation_forest_graph_results,
    custom_gae_results,
    custom_gae_pinsage_results,
    hetero_gae_results,
])

### Classification examples
Let's now look at our previous examples of reviews and how our best model (GAE with neighbor sampling) classified them.



In [ ]:
# NORMAL reviews
test_examples = yelp_chi[yelp_chi.index.isin(EXAMPLES_NORMAL)][["review", "spam"]]
test_examples["prediction"] = custom_gae_pinsage_preds[EXAMPLES_NORMAL]

for i, r in test_examples.sample(3).iterrows():
    display(HTML(f"<h4>Classified non-fraud Example #{i}</h4>"))
    display(HTML(f"<b>Classified correctly?</b> {'Yes ✅' if r['spam'] == r['prediction'] else 'No ❌'}"))
    display(HTML(f"<blockquote>{r['review']}</blockquote>"))

In [ ]:
# SPAM reviews
test_examples = yelp_chi[yelp_chi.index.isin(EXAMPLES_SPAM)][["review", "spam"]]
test_examples["prediction"] = custom_gae_pinsage_preds[EXAMPLES_SPAM]

for i, r in test_examples.sample(3).iterrows():
    display(HTML(f"<h4>Classified Fraud Example #{i}</h4>"))
    display(HTML(f"<b>Classified correctly?</b> {'Yes ✅' if r['spam'] == r['prediction'] else 'No ❌'}"))
    display(HTML(f"<blockquote>{r['review']}</blockquote>"))